In [ ]:
# This cell can be deleted in the end.
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(1, '../')

# 08 Experimental modal analysis (EMA)

In [ ]:
import pyFBS
from pyEMA import pyEMA

import numpy as np
import matplotlib.pyplot as plt
import pyvista as pv
import pandas as pd

#### 3D View
Open 3Dviewer in the background.

In [ ]:
view3D_1 = pyFBS.view3D()

Add a structure from .stl file to the 3D 

In [ ]:
stl = pyFBS.example_auto_testbench["STL"]["receiver"]
_ = view3D_1.add_stl(stl,name = "83afd2",color = "#e0e0e0",opacity = .1)

stl = pyFBS.example_auto_testbench["STL"]["transmission_mount"]
_ = view3D_1.add_stl(stl,name = "transmission_mount",color = "#83afd2",opacity = .1)

stl = pyFBS.example_auto_testbench["STL"]["roll_mount"]
_ = view3D_1.add_stl(stl,name = "roll_mount",color = "#83afd2",opacity = .1)

stl = pyFBS.example_auto_testbench["STL"]["engine_mount"]
_ = view3D_1.add_stl(stl,name = "engine_mount",color = "#83afd2",opacity = .1)

stl = pyFBS.example_auto_testbench["STL"]["ts"]
_ = view3D_1.add_stl(stl,name = "ts",color = "#d62728",opacity = .1)

stl = pyFBS.example_auto_testbench["STL"]["shaker_only"]
_ = view3D_1.add_stl(stl,name = "shaker_only",color = "#f0adad",opacity = .1)

#### Datasets

In [ ]:
pos_xlsx = pyFBS.example_auto_testbench["meas"]["xlsx_modal"]

df_acc = pd.read_excel(pos_xlsx, sheet_name='Sensors')
df_chn = pd.read_excel(pos_xlsx, sheet_name='Channels')
df_imp = pd.read_excel(pos_xlsx, sheet_name='Impacts')

### Modal analysis
First load the data

In [ ]:
# frame_rubbermounts_sourceplate
_file = pyFBS.example_auto_testbench["meas"]["Y_m_1"]
freq, Y_m1 = np.load(_file,allow_pickle = True)

# frame_rubbermounts
_file = pyFBS.example_auto_testbench["meas"]["Y_m_2"]
freq, Y_m2 = np.load(_file,allow_pickle = True)

Modeshape identification using pyEMA (LSCF/LSFD)

#### pyEMA

In [ ]:
Y =  Y_m1[:,:,0]
Y = Y.T

modal_1 = pyEMA.Model(Y,freq,pol_order_high=60,lower = 0,upper = 1000)
modal_1.get_poles()
modal_1.select_poles()

H_acc, modes_1 = modal_1.get_constants(whose_poles=modal_1,least_squares_type="old")

In [ ]:
pos_array = df_acc[["Position_1","Position_2","Position_3"]].to_numpy()*1000
faces = np.hstack([[2,1,2],[2,3,2],[2,3,13],[2,0,13],[2,0,1],[2,0,1],[2,5,1],[2,5,12],[2,4,12],[2,9,4],[2,9,8],[2,8,7],[2,0,4],[2,5,11],[2,10,11],[2,10,6],[2,2,6],[2,3,7]]).astype(np.int8)
point_cloud_1 = pv.PolyData(pos_array,faces)
pts_1 = point_cloud_1.points.copy()

_ = view3D_1.plot.add_mesh(point_cloud_1,name = "mesh",render_lines_as_tubes = True, line_width=10, color = "k",clim = [-1,1], cmap="coolwarm",scalars = np.zeros(pts_1.shape[0]),style = "wireframe")

#### Mesh animation

In [ ]:
mode_select_1 = 0
emp_1 = pyFBS.orient_in_global(modes_1[:,mode_select_1],df_chn,df_acc)

mode_dict = pyFBS.dict_animation(emp_1,"modeshape",pts = pts_1,mesh = point_cloud_1,r_scale = 50)

mode_dict["freq"] = modal_1.nat_freq[mode_select_1]
mode_dict["damp"] = modal_1.nat_xi[mode_select_1]
mode_dict["mcf"] = pyFBS.MCF(modes_1[:,mode_select_1])

view3D_1.add_modeshape(mode_dict,run_animation = True,add_note = True)

## Second part

In [ ]:
view3D_2 = pyFBS.view3D()

In [ ]:
stl = pyFBS.example_auto_testbench["STL"]["receiver"]
_ = view3D_2.add_stl(stl,name = "83afd2",color = "#e0e0e0",opacity = .1)

stl = pyFBS.example_auto_testbench["STL"]["transmission_mount"]
_ = view3D_2.add_stl(stl,name = "transmission_mount",color = "#83afd2",opacity = .1)

stl = pyFBS.example_auto_testbench["STL"]["roll_mount"]
_ = view3D_2.add_stl(stl,name = "roll_mount",color = "#83afd2",opacity = .1)

stl = pyFBS.example_auto_testbench["STL"]["engine_mount"]
_ = view3D_2.add_stl(stl,name = "engine_mount",color = "#83afd2",opacity = .1)

#### pyEMA

In [ ]:
Y =  Y_m2[:,:,0].T

modal_2 = pyEMA.Model(Y,freq,pol_order_high=60,lower = 0,upper = 1000)

modal_2 = pyEMA.Model(Y,freq,pol_order_high=60,lower = 0,upper = 1000)
modal_2.get_poles()
modal_2.select_poles()

H_acc, modes_2 = modal_2.get_constants(whose_poles=modal_2,least_squares_type="old")

In [ ]:
pos_array = df_acc[["Position_1","Position_2","Position_3"]].to_numpy()*1000

point_cloud_2 = pv.PolyData(pos_array,faces)
pts_2 = point_cloud_2.points.copy()
_ = view3D_2.plot.add_mesh(point_cloud_2,name = "mesh",render_lines_as_tubes = True, line_width=10, color = "k",clim = [-1,1], cmap="coolwarm",scalars = np.zeros(pts_2.shape[0]),style = "wireframe")

#### Mesh animation

In [ ]:
mode_select_2 = 0
emp_2 = pyFBS.orient_in_global(modes_2[:,mode_select_2],df_chn,df_acc)

mode_dict = pyFBS.dict_animation(emp_2,"modeshape",pts = pts_2,mesh = point_cloud_2,r_scale = 50)

mode_dict["freq"] = modal_2.nat_freq[mode_select_2]
mode_dict["damp"] = modal_2.nat_xi[mode_select_2]
mode_dict["mcf"] = pyFBS.MCF(modes_2[:,mode_select_2])


view3D_2.add_modeshape(mode_dict,run_animation = True,add_note = True)